# Load libraries

In [1]:
# Basic imports
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# For LLM call (placeholder)
from transformers import pipeline


# Load the Kaggle dataset

In [3]:
# Load the CSV file
# Replace the path with your actual Kaggle file location
df = pd.read_csv("/kaggle/input/datasets/sebastienverpile/consumercomplaintsdata/Consumer_Complaints.csv")

# Show first rows
df.head()


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Consumer consent provided?,Submitted via,Date sent to company,Company response to consumer,Timely response?,Consumer disputed?,Complaint ID
0,3/12/2014,Mortgage,Other mortgage,"Loan modification,collection,foreclosure",NaN,NaN,NaN,M&T BANK CORPORATION,MI,48382,NaN,NaN,Referral,3/17/2014,Closed with explanation,Yes,No,759217
1,10/1/2016,Credit reporting,NaN,Incorrect information on credit report,Account status,I have outdated information on my credit repor...,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",AL,352XX,NaN,Consent provided,Web,10/5/2016,Closed with explanation,Yes,No,2141773
2,10/17/2016,Consumer Loan,Vehicle loan,Managing the loan or lease,NaN,I purchased a new car on XXXX XXXX. The car de...,NaN,"CITIZENS FINANCIAL GROUP, INC.",PA,177XX,Older American,Consent provided,Web,10/20/2016,Closed with explanation,Yes,No,2163100
3,6/8/2014,Credit card,NaN,Bankruptcy,NaN,NaN,NaN,AMERICAN EXPRESS COMPANY,ID,83854,Older American,NaN,Web,6/10/2014,Closed with explanation,Yes,Yes,885638
4,9/13/2014,Debt collection,Credit card,Communication tactics,Frequent or repeated calls,NaN,NaN,"CITIBANK, N.A.",VA,23233,NaN,NaN,Web,9/13/2014,Closed with explanation,Yes,Yes,1027760


# Select 10–20 sentences for the knowledge base

In [4]:
# Select only the complaint text column
complaints = df["Consumer complaint narrative"].dropna().tolist()

# Keep only first 20 sentences for RAG knowledge base
knowledge_base = complaints[:20]

# Print them
for i, text in enumerate(knowledge_base):
    print(f"{i+1}. {text}\n")


1. I have outdated information on my credit report that I have previously disputed that has yet to be removed this information is more then seven years old and does not meet credit reporting requirements

2. I purchased a new car on XXXX XXXX. The car dealer called Citizens Bank to get a 10 day payoff on my loan, good till XXXX XXXX. The dealer sent the check the next day. When I balanced my checkbook on XXXX XXXX. I noticed that Citizens bank had taken the automatic payment out of my checking account at XXXX XXXX XXXX Bank. I called Citizens and they stated that they did not close the loan until XXXX XXXX. ( stating that they did not receive the check until XXXX. XXXX. ). I told them that I did not believe that the check took that long to arrive. XXXX told me a check was issued to me for the amount overpaid, they deducted additional interest. Today ( XXXX XXXX, ) I called Citizens Bank again and talked to a supervisor named XXXX, because on XXXX XXXX. I received a letter that the loan

# Convert sentences into TF‑IDF vectors (retrieval)

In [5]:
# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(stop_words="english")

# Fit on our knowledge base
kb_vectors = vectorizer.fit_transform(knowledge_base)


# Define a retrieval function

In [6]:
def retrieve(query, top_k=3):
    # Convert query to vector
    query_vec = vectorizer.transform([query])
    
    # Compute cosine similarity
    scores = cosine_similarity(query_vec, kb_vectors).flatten()
    
    # Get top-k indices
    top_indices = scores.argsort()[-top_k:][::-1]
    
    # Return retrieved sentences
    return [(knowledge_base[i], scores[i]) for i in top_indices]


# Load a small LLM (for generation)

In [7]:
# Load a small model for answering
llm = pipeline("text-generation", model="gpt2")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# Define the RAG answer function

In [8]:
def rag_answer(query):
    # Step 1: retrieve relevant sentences
    retrieved = retrieve(query, top_k=3)
    
    # Step 2: build context string
    context = "\n".join([f"- {sent}" for sent, score in retrieved])
    
    # Step 3: ask the LLM using the retrieved context
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    
    # Step 4: generate answer
    output = llm(prompt, max_length=150)[0]["generated_text"]
    
    return retrieved, output


# Ask a question

In [9]:
query = "What problems do customers report about overdraft fees?"

retrieved, answer = rag_answer(query)

print("Retrieved sentences:\n")
for sent, score in retrieved:
    print(f"- {sent}\n")

print("\nFinal Answer:\n")
print(answer)


Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Retrieved sentences:

- I am disputing the inaccurate information the Chex-Systems has on my credit report. I initially submitted a police report on XXXX/XXXX/16 and Chex Systems only deleted the items that I mentioned in the letter and not all the items that were actually listed on the police report. In other words they wanted me to say word for word to them what items were fraudulent. The total disregard of the police report and what accounts that it states that are fraudulent. If they just had paid a little closer attention to the police report I would not been in this position now and they would n't have to research once again. I would like the reported information to be removed : XXXX XXXX XXXX

- Checked my credit report after filing complaint with CFPB on XXXX. Was finally able to get access to the dispute forms and the XXXX XXXX account scheduled for deletion XX/XX/XXXX2017 was still on record. After already registering with my report number, name and social security and placin